# Experiment 9B: FPB-Only 10-Fold Cross-Validation

Train BERT-base and ModernBERT-base on identical FPB splits using 10-fold stratified CV.
Eliminates training data composition as a variable. Tests 3 configs:
- BERT-base + LoRA r=16
- ModernBERT-base + LoRA r=16
- ModernBERT-base + LoRA r=48

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn peft accelerate transformers sentencepiece protobuf scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy import stats
import json
import gc
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
NUM_FOLDS = 10
NUM_EPOCHS = 10
BATCH_SIZE = 16
LR = 2e-4
MAX_LEN = 128
LABEL_NAMES = ["negative", "neutral", "positive"]

## 1. Load FPB

In [ ]:
def load_fpb_file(agree_level):
    from huggingface_hub import hf_hub_download
    import zipfile, os
    path = hf_hub_download("financial_phrasebank", "data/FinancialPhraseBank-v1.0.zip", repo_type="dataset")
    extract_dir = "/tmp/fpb_data"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(extract_dir)
    filename = {"50agree": "Sentences_50Agree.txt", "allagree": "Sentences_AllAgree.txt"}[agree_level]
    filepath = os.path.join(extract_dir, "FinancialPhraseBank-v1.0", filename)
    sentences, labels = [], []
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            at_idx = line.rfind("@")
            if at_idx == -1:
                continue
            label_str = line[at_idx+1:].strip().lower()
            if label_str in label_map:
                sentences.append(line[:at_idx].strip())
                labels.append(label_map[label_str])
    return sentences, labels

def load_fpb(agree_level="50agree"):
    try:
        config = f"sentences_{agree_level}"
        ds = load_dataset("financial_phrasebank", config, trust_remote_code=True)["train"]
        return list(ds["sentence"]), list(ds["label"])
    except Exception:
        return load_fpb_file(agree_level)

texts, labels = load_fpb("50agree")

print(f"FPB total: {len(texts)} samples")
print(f"Label distribution: {pd.Series(labels).value_counts().sort_index().to_dict()}")

## 2. Model Configs

In [ ]:
MODELS = {
    "bert-base-r16": {
        "name": "bert-base-uncased",
        "lora_targets": ["query", "key", "value", "dense"],
        "lora_r": 16,
        "lora_alpha": 32,
    },
    "modernbert-base-r16": {
        "name": "answerdotai/ModernBERT-base",
        "lora_targets": ["Wqkv", "out_proj", "Wi", "Wo"],
        "lora_r": 16,
        "lora_alpha": 32,
    },
    "modernbert-base-r48": {
        "name": "answerdotai/ModernBERT-base",
        "lora_targets": ["Wqkv", "out_proj", "Wi", "Wo"],
        "lora_r": 48,
        "lora_alpha": 96,
    },
}

## 3. Training Function

In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, lbls):
        self.encodings = encodings
        self.labels = lbls
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    acc = accuracy_score(eval_pred.label_ids, preds)
    f1 = f1_score(eval_pred.label_ids, preds, average="macro")
    return {"accuracy": acc, "macro_f1": f1}

def train_and_evaluate(model_key, train_idx, val_idx, fold_num):
    cfg = MODELS[model_key]
    tokenizer = AutoTokenizer.from_pretrained(cfg["name"])
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["name"], num_labels=3
    )

    lora_config = LoraConfig(
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        target_modules=cfg["lora_targets"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(model, lora_config)
    if fold_num == 0:
        model.print_trainable_parameters()

    train_texts_fold = [texts[i] for i in train_idx]
    train_labels_fold = [labels[i] for i in train_idx]
    val_texts_fold = [texts[i] for i in val_idx]
    val_labels_fold = [labels[i] for i in val_idx]

    train_enc = tokenizer(train_texts_fold, truncation=True, padding=True, max_length=MAX_LEN)
    val_enc = tokenizer(val_texts_fold, truncation=True, padding=True, max_length=MAX_LEN)

    train_ds = SentimentDataset(train_enc, train_labels_fold)
    val_ds = SentimentDataset(val_enc, val_labels_fold)

    args = TrainingArguments(
        output_dir=f"cv_output/{model_key}/fold_{fold_num}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.001,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=50,
        seed=SEED,
        fp16=True,
        report_to="none",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()
    preds = trainer.predict(val_ds)
    pred_labels = np.argmax(preds.predictions, axis=-1)

    report = classification_report(
        val_labels_fold, pred_labels,
        target_names=LABEL_NAMES, output_dict=True
    )

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "fold": fold_num,
        "model": model_key,
        "accuracy": results["eval_accuracy"],
        "macro_f1": results["eval_macro_f1"],
        "eval_loss": results["eval_loss"],
        "neg_f1": report["negative"]["f1-score"],
        "neu_f1": report["neutral"]["f1-score"],
        "pos_f1": report["positive"]["f1-score"],
    }

## 4. Run 10-Fold CV

In [ ]:
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
all_results = []

for model_key in MODELS:
    print(f"\n{'='*60}")
    print(f"MODEL: {model_key}")
    print(f"{'='*60}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
        print(f"\n--- Fold {fold+1}/{NUM_FOLDS} (train={len(train_idx)}, val={len(val_idx)}) ---")
        result = train_and_evaluate(model_key, train_idx, val_idx, fold)
        all_results.append(result)
        print(f"  Acc: {result['accuracy']:.4f}, F1: {result['macro_f1']:.4f}")

with open("cv_results_09b.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nAll results saved to cv_results_09b.json")

## 5. Summary & Statistical Tests

In [ ]:
df = pd.DataFrame(all_results)

summary = df.groupby("model").agg(
    mean_acc=("accuracy", "mean"),
    std_acc=("accuracy", "std"),
    mean_f1=("macro_f1", "mean"),
    std_f1=("macro_f1", "std"),
).round(4)

print("\n10-FOLD CV RESULTS (FPB sentences_50agree)")
print("=" * 70)
print(summary.to_string())
print()

bert_accs = df[df.model == "bert-base-r16"]["accuracy"].values
mb16_accs = df[df.model == "modernbert-base-r16"]["accuracy"].values
mb48_accs = df[df.model == "modernbert-base-r48"]["accuracy"].values

t1, p1 = stats.ttest_rel(bert_accs, mb16_accs)
t2, p2 = stats.ttest_rel(bert_accs, mb48_accs)
t3, p3 = stats.ttest_rel(mb48_accs, mb16_accs)

print(f"Paired t-test BERT r16 vs ModernBERT r16: t={t1:.3f}, p={p1:.4f} {'*' if p1 < 0.05 else ''}")
print(f"Paired t-test BERT r16 vs ModernBERT r48: t={t2:.3f}, p={p2:.4f} {'*' if p2 < 0.05 else ''}")
print(f"Paired t-test ModernBERT r48 vs r16:      t={t3:.3f}, p={p3:.4f} {'*' if p3 < 0.05 else ''}")
print()
print(f"Mean gap (BERT - MB r16): {(bert_accs.mean() - mb16_accs.mean())*100:.2f}pp")
print(f"Mean gap (BERT - MB r48): {(bert_accs.mean() - mb48_accs.mean())*100:.2f}pp")

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for model_key in MODELS:
    model_df = df[df.model == model_key]
    ax.plot(range(1, NUM_FOLDS+1), model_df["accuracy"].values, 'o-', label=model_key)
ax.set_xlabel("Fold")
ax.set_ylabel("Accuracy")
ax.set_title("Per-Fold Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
model_names = list(MODELS.keys())
accs_by_model = [df[df.model == m]["accuracy"].values for m in model_names]
bp = ax.boxplot(accs_by_model, labels=model_names, patch_artist=True)
colors = ["#4CAF50", "#2196F3", "#FF9800"]
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy Distribution (10 folds)")
ax.grid(True, alpha=0.3)

plt.suptitle("Experiment 9B: FPB 10-Fold CV — BERT vs ModernBERT", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("09b_fpb_crossval.png", dpi=150, bbox_inches="tight")
plt.show()